# 🧠 Distillation DeepSeek → T5-base

**Principe** : DeepSeek (professeur) traduit les problèmes GSM8K en opérations structurées. T5-base (étudiant) apprend cette traduction.

**Prérequis** :
1. Uploader `deepseek_distill_train_compact.json` via **+ Add input**
2. Activer **GPU T4 x2** (Settings → Accelerator)
3. Exécuter tout

**Sortie** : `/kaggle/working/t5_base_kaggle/final/` → télécharger

In [ ]:
#!/usr/bin/env python3"""kaggle_train_t5.py — Entraînement T5-base sur GPU Kaggle (T4)================================================================Distillation : DeepSeek (professeur) → T5-base (étudiant).UTILISATION SUR KAGGLE :1. Créer un notebook Kaggle (GPU T4 x2 activé)2. Uploader data/deepseek_distill_train_compact.json via "+ Add input"3. Copier ce script dans une cellule et l'exécuter4. Le modèle est sauvegardé dans /kaggle/working/ → téléchargeableLE MODÈLE RÉSULTANT :  data/t5_base_kaggle/final/  (adaptateur LoRA + tokenizer)  → à télécharger et placer dans engine/data/t5_base_kaggle/"""import sys, os, json, time, gc# ═══════════════════════════════════════════════════════════════════════════# 1. CHARGEMENT DES DONNÉES# ═══════════════════════════════════════════════════════════════════════════def find_dataset():    """Cherche le dataset dans les emplacements Kaggle possibles."""    candidates = [        '/kaggle/input/deepseek-distill/deepseek_distill_train_compact.json',        '/kaggle/input/deepseek_distill_train_compact.json',        '/kaggle/working/deepseek_distill_train_compact.json',        'data/deepseek_distill_train_compact.json',        'data/deepseek_distill_train.json',    ]    for path in candidates:        if os.path.exists(path):            return path    # Chercher récursivement dans /kaggle/input    if os.path.exists('/kaggle/input'):        for root, dirs, files in os.walk('/kaggle/input'):            for f in files:                if f.endswith('.json') and 'deepseek' in f.lower():                    return os.path.join(root, f)    raise FileNotFoundError(        "Dataset non trouvé. Uploader deepseek_distill_train_compact.json "        "via '+ Add input' dans Kaggle.")print("═══ DISTILLATION T5-BASE SUR KAGGLE ═══\n")print(f"GPU disponible : {__import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'CPU'}")data_path = find_dataset()print(f"Dataset : {data_path}")import jsonwith open(data_path, 'r', encoding='utf-8') as f:    data = json.load(f)# Filtrer les entrées videsdata = [d for d in data if d['output'].strip()]print(f"Données : {len(data)} paires problème→opérations")# Split train/val (95/5)import randomrandom.seed(42)random.shuffle(data)val_size = max(1, int(len(data) * 0.05))train_data = data[val_size:]val_data = data[:val_size]print(f"Train : {len(train_data)} | Val : {len(val_data)}")# ═══════════════════════════════════════════════════════════════════════════# 2. MODÈLE T5-BASE + LoRA# ═══════════════════════════════════════════════════════════════════════════import torchfrom transformers import (    AutoTokenizer, AutoModelForSeq2SeqLM,    TrainingArguments, Trainer, DataCollatorForSeq2Seq,)from datasets import Datasetfrom peft import LoraConfig, get_peft_model, TaskTypeMODEL_NAME = 'google/flan-t5-base'  # 250M params — parfait pour T4 16GBprint(f"\nChargement de {MODEL_NAME}...")tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)# LoRA — seulement 1% des paramètres entraînéslora_config = LoraConfig(    task_type=TaskType.SEQ_2_SEQ_LM,    r=16, lora_alpha=32, lora_dropout=0.1,    target_modules=["q", "v"],)model = get_peft_model(model, lora_config)model.print_trainable_parameters()def preprocess(examples):    inputs = ["translate to operations: " + t for t in examples["input"]]    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)    labels = tokenizer(text_target=examples["output"], max_length=256,                      truncation=True, padding=False)    model_inputs["labels"] = labels["input_ids"]    return model_inputstrain_hf = Dataset.from_list(train_data)val_hf = Dataset.from_list(val_data)train_hf = train_hf.map(preprocess, batched=True, remove_columns=train_hf.column_names)val_hf = val_hf.map(preprocess, batched=True, remove_columns=val_hf.column_names)# ═══════════════════════════════════════════════════════════════════════════# 3. ENTRAÎNEMENT (GPU)# ═══════════════════════════════════════════════════════════════════════════training_args = TrainingArguments(    output_dir='/kaggle/working/t5_base_kaggle',    num_train_epochs=8,    per_device_train_batch_size=8,    per_device_eval_batch_size=8,    gradient_accumulation_steps=2,   # batch effectif = 16    learning_rate=3e-4,    warmup_ratio=0.1,    weight_decay=0.01,    logging_dir='/kaggle/working/logs',    logging_steps=25,    eval_strategy="epoch",    save_strategy="epoch",    save_total_limit=2,    load_best_model_at_end=True,    metric_for_best_model="eval_loss",    greater_is_better=False,    fp16=True,                        # GPU T4 → mixed precision    report_to="none",)trainer = Trainer(    model=model,    args=training_args,    train_dataset=train_hf,    eval_dataset=val_hf,    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),)print(f"\nDémarrage entraînement (8 époques, GPU T4, batch effectif=16)...")t0 = time.time()trainer.train()print(f"✓ Entraînement terminé en {(time.time()-t0)/60:.1f} min")# ═══════════════════════════════════════════════════════════════════════════# 4. SAUVEGARDE + TEST# ═══════════════════════════════════════════════════════════════════════════final_path = '/kaggle/working/t5_base_kaggle/final'model.save_pretrained(final_path)tokenizer.save_pretrained(final_path)print(f"✓ Modèle sauvegardé : {final_path}")# Test rapideprint("\n═══ TEST RAPIDE ═══")from peft import PeftModelbase = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)eval_model = PeftModel.from_pretrained(base, final_path)eval_model.eval().to('cuda')tests = [    ("John has 5 apples. He buys 3 more. How many apples does he have?", "8"),    ("Mary had 10 cookies. She ate 4. How many cookies does she have left?", "6"),    ("There are 6 boxes. Each box has 5 pencils. How many pencils are there in total?", "30"),    ("John has 5 apples. Mary has 3 times as many. How many apples does Mary have?", "15"),    ("James earns 20 dollars per hour. He works 8 hours. How much does he earn?", "160"),]for q, expected in tests:    inputs = tokenizer("translate to operations: " + q,                      return_tensors="pt", truncation=True, max_length=512).to('cuda')    with torch.no_grad():        outputs = eval_model.generate(**inputs, max_new_tokens=256, num_beams=3)    texte = tokenizer.decode(outputs[0], skip_special_tokens=True)    print(f"\nQ: {q[:60]}...")    print(f"  → {texte[:200]}")print("\n═══ TERMINÉ ═══")print("Télécharger /kaggle/working/t5_base_kaggle/final/")print("→ placer dans engine/data/t5_base_kaggle/")print("→ utiliser : DistilledSolver(model_path='data/t5_base_kaggle/final',")print("                          base_model='google/flan-t5-base')")